# 20260923 Visualize Core

Use this notebook for full-session checks: behavior in the arena, arena/startbox ROI overlays, head direction, and optional neural maps for curated cells. Keep trial-by-trial comparisons in `20260923_visualize_trials.ipynb`.

## Imports

In [ ]:
from pathlib import Path
import os
import re
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from preprocess_functions.manifest import (
    EXPERIMENT_METADATA_COLUMNS,
    experiment_metadata_from_record,
    load_session_records,
)
from preprocess_functions.pipeline import default_aligned_session_path, default_cue_events_path, default_trials_path
from preprocess_functions import plot, roi

## Config

In [ ]:
OUTPUT_ROOT = Path("preprocess_out")
MANIFEST = OUTPUT_ROOT / "manifest_with_cells.csv"
if not MANIFEST.exists():
    MANIFEST = OUTPUT_ROOT / "manifest_with_h5.csv"
if not MANIFEST.exists():
    MANIFEST = Path("data_paths/RSC_PPC_Cohort1_paths.xlsx")

SHEET = None
LAB_DRIVE = os.environ.get("LAB_DRIVE_PATH") or None
RECORDING_ID = None
FPS = 30.0

ROI_NAMES = ["arena", "startbox_L", "startbox_R"]
LOAD_OR_COLLECT_ROIS = True
SAVE_ROI_FEATURES_TO_ALIGNED_CSV = False

TRAJECTORY_DOWNSAMPLE = 5
HD_DOWNSAMPLE = 20
CELL_COL = None

## Load Recording

In [ ]:
records = load_session_records(MANIFEST, sheet_name=SHEET, lab_drive=LAB_DRIVE)
records_df = pd.DataFrame([
    {
        "recording_id": record.recording_id,
        "session_id": record.session_id,
        "mouse_id": record.mouse_id,
        "trial_type": record.trial_type,
        **experiment_metadata_from_record(record),
        "aligned_csv": str(default_aligned_session_path(OUTPUT_ROOT, record)),
        "cell_csv": str(record.cell_csv) if record.cell_csv else None,
    }
    for record in records
])
display(records_df)

In [ ]:
def choose_record(records, recording_id=None):
    if recording_id is None:
        return records[0]
    for record in records:
        if recording_id in {record.recording_id, record.session_id, record.trial_type, record.mouse_id}:
            return record
    raise ValueError(f"No manifest row matched {recording_id!r}.")


record = choose_record(records, RECORDING_ID)
aligned_csv = default_aligned_session_path(OUTPUT_ROOT, record)
if not aligned_csv.exists():
    raise FileNotFoundError(f"Missing aligned CSV. Run build_aligned_sessions first: {aligned_csv}")

df = pd.read_csv(aligned_csv)
print("recording_id:", record.recording_id)
print("aligned CSV:", aligned_csv)
print("shape:", df.shape)
display(df.head())

## Optional: Load And Stitch Split Recordings

Use this when split videos were processed as separate `recording_id` rows, but you want to inspect two or more splits as one analysis block. This does not replace the current `df` unless you uncomment the handoff line at the bottom.

In [ ]:
# Fill this only when you want to compare/stitch processed split recordings.
# Each key becomes an analysis_block. Values must match records_df["recording_id"].
ANALYSIS_BLOCKS = {
    # "FE1_VC_Sal": [
    #     "C5731A_20260901_FE1_VC_Sal_1",
    #     "C5731A_20260901_FE1_VC_Sal_2",
    # ],
    # "FE2_VC_Sal": [
    #     "C5731A_20260901_FE2_VC_Sal_1",
    #     "C5731A_20260901_FE2_VC_Sal_2",
    # ],
}

# Set this to one key from ANALYSIS_BLOCKS to make downstream cells plot that
# stitched block as one long session. Leave as None for normal single-recording plots.
STITCHED_BLOCK_TO_USE = None


def load_aligned_recording(record):
    path = default_aligned_session_path(OUTPUT_ROOT, record)
    if not path.exists():
        raise FileNotFoundError(f"Missing aligned CSV for {record.recording_id}: {path}")
    df_i = pd.read_csv(path)
    df_i["source_recording_id"] = record.recording_id
    df_i["recording_id"] = record.recording_id
    df_i["session_id"] = record.session_id
    df_i["mouse_id"] = record.mouse_id
    df_i["trial_type"] = record.trial_type
    for col, value in experiment_metadata_from_record(record).items():
        df_i[col] = value
    df_i["aligned_csv"] = str(path)
    return df_i


records_by_id = {record.recording_id: record for record in records}
split_session_dfs = {}
stitched_session_dfs = {}
compare_df = pd.DataFrame()

for block_name, recording_ids in ANALYSIS_BLOCKS.items():
    pieces = []
    for split_order, recording_id in enumerate(recording_ids):
        if recording_id not in records_by_id:
            raise KeyError(f"{recording_id!r} is not in records_df['recording_id']")
        df_i = load_aligned_recording(records_by_id[recording_id]).copy()
        df_i["analysis_block"] = block_name
        df_i["split_order"] = split_order
        split_session_dfs[recording_id] = df_i
        pieces.append(df_i)

    if pieces:
        block_df = pd.concat(pieces, ignore_index=True)
        block_df["analysis_frame"] = np.arange(len(block_df))
        block_df["analysis_time_s"] = block_df["analysis_frame"] / FPS if FPS else np.nan
        block_df["analysis_source_frame"] = block_df.groupby("source_recording_id").cumcount()
        stitched_session_dfs[block_name] = block_df

if stitched_session_dfs:
    compare_df = pd.concat(stitched_session_dfs.values(), ignore_index=True)
    summary = (
        compare_df
        .groupby(["analysis_block", "source_recording_id"], sort=False)
        .agg(
            n_frames=("analysis_frame", "size"),
            start_analysis_frame=("analysis_frame", "min"),
            end_analysis_frame=("analysis_frame", "max"),
            trial_type=("trial_type", "first"),
        )
        .reset_index()
    )
    display(summary)
    display(compare_df.head())
else:
    print("ANALYSIS_BLOCKS is empty. Add recording IDs above when you need stitched split-session views.")

if STITCHED_BLOCK_TO_USE is not None:
    if STITCHED_BLOCK_TO_USE not in stitched_session_dfs:
        raise KeyError(f"{STITCHED_BLOCK_TO_USE!r} is not in stitched_session_dfs")
    df = stitched_session_dfs[STITCHED_BLOCK_TO_USE].copy()
    first_recording_id = str(df["source_recording_id"].iloc[0])
    record = records_by_id[first_recording_id]
    aligned_csv = Path(df["aligned_csv"].iloc[0])
    print("Using stitched block for downstream plots:", STITCHED_BLOCK_TO_USE)
    print("Frames:", len(df), "Duration min:", round(float(df["analysis_time_s"].max() / 60), 2))
    print("Time/color column for long-session plots: analysis_frame")
    display(df[["analysis_block", "source_recording_id", "analysis_frame", "analysis_time_s"]].head())


## Optional: Population / Manifold Loader

Use this when you need many processed recordings available for population analyses. The registry is cheap to build; loading every aligned CSV can be memory-heavy, so flip `POPULATION_LOAD_DATAFRAMES` only when you are ready.

In [ ]:
# Empty filters mean "include everything available".
POPULATION_RECORDING_IDS = []
POPULATION_MOUSE_IDS = []
POPULATION_TRIAL_TYPES = []  # Optional legacy filter; prefer metadata filters below for this project.
POPULATION_MANIPULATIONS = []
POPULATION_CUE_CONDITIONS = []
POPULATION_CUE_ROTATION_DEG = []
POPULATION_VISUAL_CUES_PRESENT = []

# For cross-session manifold work, registered_cell_* columns give shared neuron axes.
# Set False only for within-recording analyses where local cell_* columns are okay.
POPULATION_REQUIRE_REGISTERED = True

# Keep False while exploring the registry; set True when you want to actually read all CSVs.
POPULATION_LOAD_DATAFRAMES = False
POPULATION_CONCATENATE_COMMON_CELLS = False
POPULATION_SAMPLE_EVERY_N_FRAMES = 1
POPULATION_MAX_RECORDINGS = None

POPULATION_META_COLS = [
    "recording_id", "session_id", "source_recording_id", "mouse_id", "trial_type",
    *EXPERIMENT_METADATA_COLUMNS,
    "global_idx", "global_ts", "analysis_frame", "analysis_time_s",
    "beh_frame_idx", "neu_frame_idx", "cell_frame_idx",
    "ear_mid_x", "ear_mid_y", "head_dir_rad",
    "in_arena", "arena_only", "video_any_port_active", "bpod_any_port_active",
]


def natural_sort_key(text):
    parts = re.split(r"(\d+)", str(text))
    return [int(part) if part.isdigit() else part for part in parts]


def population_neural_columns(df_i, require_registered=True):
    registered = sorted(
        [col for col in df_i.columns if col.startswith("registered_cell_")],
        key=natural_sort_key,
    )
    if registered:
        return registered
    if require_registered:
        return []
    return sorted([col for col in df_i.columns if col.startswith("cell_")], key=natural_sort_key)


population_registry = records_df.copy()
population_registry["aligned_exists"] = population_registry["aligned_csv"].map(lambda path: Path(path).exists())
population_registry = population_registry[population_registry["aligned_exists"]].copy()

if POPULATION_RECORDING_IDS:
    population_registry = population_registry[population_registry["recording_id"].isin(POPULATION_RECORDING_IDS)]
if POPULATION_MOUSE_IDS:
    population_registry = population_registry[population_registry["mouse_id"].isin(POPULATION_MOUSE_IDS)]
if POPULATION_TRIAL_TYPES:
    population_registry = population_registry[population_registry["trial_type"].isin(POPULATION_TRIAL_TYPES)]
if POPULATION_MANIPULATIONS and "manipulation" in population_registry.columns:
    population_registry = population_registry[population_registry["manipulation"].isin(POPULATION_MANIPULATIONS)]
if POPULATION_CUE_CONDITIONS and "cue_condition" in population_registry.columns:
    population_registry = population_registry[population_registry["cue_condition"].isin(POPULATION_CUE_CONDITIONS)]
if POPULATION_CUE_ROTATION_DEG and "cue_rotation_deg" in population_registry.columns:
    cue_rotation = pd.to_numeric(population_registry["cue_rotation_deg"], errors="coerce")
    population_registry = population_registry[cue_rotation.isin(POPULATION_CUE_ROTATION_DEG)]
if POPULATION_VISUAL_CUES_PRESENT and "visual_cues_present" in population_registry.columns:
    population_registry = population_registry[population_registry["visual_cues_present"].isin(POPULATION_VISUAL_CUES_PRESENT)]
if POPULATION_MAX_RECORDINGS is not None:
    population_registry = population_registry.head(int(POPULATION_MAX_RECORDINGS)).copy()

registry_display_cols = [
    "recording_id", "session_id", "mouse_id", "trial_type",
    "manipulation", "cue_condition", "cue_rotation_deg", "visual_cues_present",
    "aligned_csv",
]
display(population_registry[[col for col in registry_display_cols if col in population_registry.columns]])


def load_population_aligned_data(registry):
    population_dfs = {}
    load_rows = []
    cell_sets = {}

    for _, row in registry.iterrows():
        recording_id = str(row["recording_id"])
        path = Path(row["aligned_csv"])
        df_i = pd.read_csv(path)

        if POPULATION_SAMPLE_EVERY_N_FRAMES > 1:
            df_i = df_i.iloc[:: int(POPULATION_SAMPLE_EVERY_N_FRAMES)].copy()

        df_i["source_recording_id"] = recording_id
        df_i["recording_id"] = recording_id
        df_i["session_id"] = row.get("session_id")
        df_i["mouse_id"] = row.get("mouse_id")
        df_i["trial_type"] = row.get("trial_type")
        for col in EXPERIMENT_METADATA_COLUMNS:
            if col in registry.columns:
                df_i[col] = row.get(col)
        df_i["analysis_frame"] = np.arange(len(df_i))
        df_i["analysis_time_s"] = df_i["analysis_frame"] / FPS if FPS else np.nan

        neural_cols = population_neural_columns(df_i, require_registered=POPULATION_REQUIRE_REGISTERED)
        if not neural_cols:
            load_rows.append({
                "recording_id": recording_id,
                "status": "no_registered_cells" if POPULATION_REQUIRE_REGISTERED else "no_cells",
                "n_frames": len(df_i),
                "n_neural_cols": 0,
                "memory_mb": np.nan,
            })
            continue

        keep_cols = [col for col in POPULATION_META_COLS if col in df_i.columns]
        keep_cols = list(dict.fromkeys([*keep_cols, *neural_cols]))
        df_i = df_i[keep_cols].copy()

        for col in neural_cols:
            df_i[col] = pd.to_numeric(df_i[col], errors="coerce").astype("float32")

        population_dfs[recording_id] = df_i
        cell_sets[recording_id] = set(neural_cols)
        load_rows.append({
            "recording_id": recording_id,
            "status": "loaded",
            "n_frames": len(df_i),
            "n_neural_cols": len(neural_cols),
            "memory_mb": round(df_i.memory_usage(deep=True).sum() / 1_000_000, 2),
        })

    if cell_sets:
        common_cells = sorted(set.intersection(*cell_sets.values()), key=natural_sort_key)
        union_cells = sorted(set.union(*cell_sets.values()), key=natural_sort_key)
    else:
        common_cells = []
        union_cells = []

    return population_dfs, pd.DataFrame(load_rows), common_cells, union_cells


population_dfs = {}
population_load_summary = pd.DataFrame()
common_population_cells = []
union_population_cells = []
population_common_df = pd.DataFrame()

if POPULATION_LOAD_DATAFRAMES:
    (
        population_dfs,
        population_load_summary,
        common_population_cells,
        union_population_cells,
    ) = load_population_aligned_data(population_registry)

    display(population_load_summary)
    print("loaded recordings:", len(population_dfs))
    print("common neural columns:", len(common_population_cells))
    print("union neural columns:", len(union_population_cells))

    if POPULATION_CONCATENATE_COMMON_CELLS and population_dfs:
        if not common_population_cells:
            raise ValueError("No neural columns are common to every loaded recording.")
        frames = []
        for recording_id, df_i in population_dfs.items():
            meta_cols = [col for col in POPULATION_META_COLS if col in df_i.columns]
            frames.append(df_i[list(dict.fromkeys([*meta_cols, *common_population_cells]))])
        population_common_df = pd.concat(frames, ignore_index=True)
        print("population_common_df shape:", population_common_df.shape)
else:
    print("Set POPULATION_LOAD_DATAFRAMES = True to read aligned CSVs into population_dfs.")


## Quick Selection And Plot Recipes

Use this as a scratchpad for selecting a few recordings by metadata, making a stitched block, and plotting population/cell summaries on the fly.

In [ ]:
import matplotlib.pyplot as plt


def select_recording_ids(
    *,
    mouse_id=None,
    session_id=None,
    manipulation=None,
    cue_condition=None,
    cue_rotation_deg=None,
    visual_cues_present=None,
    contains=None,
):
    table = records_df.copy()
    filters = {
        "mouse_id": mouse_id,
        "session_id": session_id,
        "manipulation": manipulation,
        "cue_condition": cue_condition,
        "visual_cues_present": visual_cues_present,
    }
    for col, value in filters.items():
        if value is None or col not in table.columns:
            continue
        values = value if isinstance(value, (list, tuple, set)) else [value]
        table = table[table[col].isin(values)]

    if cue_rotation_deg is not None and "cue_rotation_deg" in table.columns:
        values = cue_rotation_deg if isinstance(cue_rotation_deg, (list, tuple, set)) else [cue_rotation_deg]
        rotation = pd.to_numeric(table["cue_rotation_deg"], errors="coerce")
        table = table[rotation.isin(values)]

    if contains:
        table = table[table["recording_id"].astype(str).str.contains(str(contains), case=False, na=False)]

    display_cols = [
        "recording_id", "session_id", "mouse_id", "trial_type",
        "manipulation", "cue_condition", "cue_rotation_deg", "visual_cues_present",
    ]
    display(table[[col for col in display_cols if col in table.columns]])
    return table["recording_id"].tolist()


def stitch_recordings_for_plot(block_name, recording_ids):
    pieces = []
    for split_order, recording_id in enumerate(recording_ids):
        if recording_id not in records_by_id:
            raise KeyError(f"{recording_id!r} is not in records_df['recording_id']")
        df_i = load_aligned_recording(records_by_id[recording_id]).copy()
        df_i["analysis_block"] = block_name
        df_i["split_order"] = split_order
        pieces.append(df_i)

    stitched = pd.concat(pieces, ignore_index=True)
    stitched["analysis_frame"] = np.arange(len(stitched))
    stitched["analysis_time_s"] = stitched["analysis_frame"] / FPS if FPS else np.nan
    stitched["analysis_source_frame"] = stitched.groupby("source_recording_id").cumcount()
    return stitched


def neural_columns(dataframe, require_registered=True):
    registered = sorted(
        [col for col in dataframe.columns if col.startswith("registered_cell_")],
        key=natural_sort_key,
    )
    if registered or require_registered:
        return registered
    return sorted([col for col in dataframe.columns if col.startswith("cell_")], key=natural_sort_key)


def plot_population_heatmap(dataframe=None, cell_cols=None, every=1, zscore=True, title=None):
    source = dataframe
    if source is None:
        source = population_common_df if not population_common_df.empty else df
    if cell_cols is None:
        cell_cols = common_population_cells if common_population_cells else neural_columns(source, require_registered=False)
    if not cell_cols:
        raise ValueError("No neural columns found for heatmap.")

    X = source[cell_cols].iloc[::every].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    if zscore:
        X = (X - np.nanmean(X, axis=0)) / np.nanstd(X, axis=0)

    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(X.T, aspect="auto", cmap="viridis", vmin=-2 if zscore else None, vmax=2 if zscore else None)
    ax.set_xlabel("frame")
    ax.set_ylabel("registered neuron" if any(str(col).startswith("registered_cell_") for col in cell_cols) else "neuron")
    ax.set_title(title or "Population activity")
    plt.colorbar(im, ax=ax, label="z activity" if zscore else "activity")
    plt.show()
    return fig, ax


def quick_cell_plots(dataframe=None, cell_col=None):
    source = dataframe if dataframe is not None else df
    cells = neural_columns(source, require_registered=False)
    if cell_col is None:
        if not cells:
            raise ValueError("No neural columns found.")
        cell_col = cells[0]
    print("cell:", cell_col)
    if arena_mask is not None:
        plot.plot_2d_ratemap(source, cell_col=cell_col, arena_mask=arena_mask, bins=20)
    if "head_dir_rad" in source.columns:
        plot.plot_hd(source, cell_col=cell_col, head_dir_col="head_dir_rad", n_bins=36)
    if arena_mask is not None and {"ear_mid_x", "ear_mid_y", "nose.x", "nose.y", "head_dir_rad"}.issubset(source.columns):
        plot.plot_ebc_heatmap(source, arena_mask=arena_mask, cell_col=cell_col, frame_stride=3)


# Examples:
# ids = select_recording_ids(mouse_id="C5731A", manipulation="CNO")
# ids = select_recording_ids(mouse_id="C5731A", cue_condition="rotated_cues", cue_rotation_deg=90)
# long_df = stitch_recordings_for_plot("CNO_day", ids[:2])
# plot_population_heatmap(long_df, every=5, title="CNO stitched population")
# quick_cell_plots(long_df, cell_col="registered_cell_0")


## Arena ROIs

In [ ]:
def choose_roi_id(record, roi_names, root=PROJECT_ROOT):
    for candidate in [record.recording_id, record.session_id]:
        if candidate and any(roi.roi_json_path(candidate, name, root=root).exists() for name in roi_names):
            return candidate
    return record.recording_id


def behavior_video_path(record, df):
    if "beh_vid_path" in df.columns and df["beh_vid_path"].notna().any():
        values = df["beh_vid_path"].dropna().astype(str).str.strip()
        values = values[~values.str.lower().isin(["", "nan", "none", "null"])]
        if not values.empty:
            return Path(values.iloc[0])
    if record.beh_vid is not None:
        return Path(record.beh_vid)
    raise FileNotFoundError("No behavior video path found for ROI selection.")


masks = {}
arena_mask = None
roi_paths = {}

if LOAD_OR_COLLECT_ROIS:
    ROI_ID = choose_roi_id(record, ROI_NAMES)
    video_path = behavior_video_path(record, df)
    print("behavior video:", video_path)

    rois, roi_paths = roi.load_or_collect_named_rois(
        video_path=video_path,
        session_id=ROI_ID,
        roi_names=ROI_NAMES,
        root=PROJECT_ROOT,
        folder_name="arena_rois",
    )
    height, width = roi.get_video_hw(video_path)
    masks = roi.build_roi_masks(rois, height, width)
    arena_mask = masks.get("arena")

    if {"ear_mid_x", "ear_mid_y"}.issubset(df.columns):
        df = roi.add_roi_features(df, rois, ref_x="ear_mid_x", ref_y="ear_mid_y")
        df = roi.add_arena_only_column(df)
        if SAVE_ROI_FEATURES_TO_ALIGNED_CSV:
            aligned_csv.parent.mkdir(parents=True, exist_ok=True)
            df.to_csv(aligned_csv, index=False)
            print("updated aligned CSV:", aligned_csv)

    print("ROI JSONs:")
    for name, path in roi_paths.items():
        print(f"  {name}: {path}")
else:
    print("ROI loading disabled.")

## Core Behavior Plots

In [ ]:
TIME_PROGRESS_COL = "analysis_frame" if "analysis_frame" in df.columns else "global_idx"
print("time/color column:", TIME_PROGRESS_COL)

plot.plot_trajectory(
    df,
    masks=masks or None,
    x_col="ear_mid_x",
    y_col="ear_mid_y",
    ts_col=TIME_PROGRESS_COL,
    downsample=TRAJECTORY_DOWNSAMPLE,
)

In [ ]:
TIME_PROGRESS_COL = "analysis_frame" if "analysis_frame" in df.columns else "global_idx"

if "head_dir_rad" in df.columns:
    plot.plot_hd_trajectory(
        df,
        masks=masks or None,
        x_col="ear_mid_x",
        y_col="ear_mid_y",
        angle_col="head_dir_rad",
        ts_col=TIME_PROGRESS_COL,
        downsample=HD_DOWNSAMPLE,
    )

## Optional Neural Overview

In [ ]:
cell_cols = [col for col in df.columns if col.startswith("registered_cell_")]
if not cell_cols:
    cell_cols = [col for col in df.columns if col.startswith("cell_")]
if CELL_COL is None and cell_cols:
    CELL_COL = cell_cols[0]
print("available cells:", len(cell_cols))
print("selected cell:", CELL_COL)

In [ ]:
if CELL_COL is not None and arena_mask is not None:
    plot.plot_2d_ratemap(
        df,
        cell_col=CELL_COL,
        arena_mask=arena_mask,
        bins=20,
    )

In [ ]:
if CELL_COL is not None and "head_dir_rad" in df.columns:
    plot.plot_hd(df, cell_col=CELL_COL, head_dir_col="head_dir_rad", n_bins=36)

In [ ]:
required_for_ebc = {"ear_mid_x", "ear_mid_y", "nose.x", "nose.y", "head_dir_rad"}
if CELL_COL is not None and arena_mask is not None and required_for_ebc.issubset(df.columns):
    plot.plot_cell_summary(
        df,
        cell_col=CELL_COL,
        arena_mask=arena_mask,
        egocentric_smooth_sigma=(1.0, 1.5),
        head_direction_bins=36,
    )